# 스케줄러에 맡기기 — 문서를 넣으면 답이 바뀌기까지

**「코랩으로 만드는 업무 챗봇」 18장 예제** — 파이프라인(kb_01)과 답하기(kb_02)를 잇고 **스케줄러에 맡겨**, 문서를 넣으면 챗봇의 답이 바뀌는 흐름을 끝에서 끝으로 돌립니다.

- 걸리는 시간: 약 40분
- 키: 필요 없습니다(전부 키 없이 끝까지 돕니다).

### 시작하는 방법
1. 「Drive로 복사」를 누른 뒤 위에서부터 한 칸씩 실행합니다(칸 왼쪽 ▶ 또는 Shift+Enter).
2. 「준비」 아래의 칸들은 처음 한 번만 실행합니다. 설치는 1~2분 걸립니다.
3. 예제마다 **확인 포인트**와 **바꿔 보기**가 있습니다. 확인 포인트의 값이 화면과 같은지 보고, 바꿔 보기는 직접 고쳐 다시 실행합니다.

### 이 노트북의 환경
실제 서비스에서는 Airflow 를 서버에 따로 띄웁니다. 이 노트북은 같은 Airflow(3.3.2)를 코랩 안 가상환경 하나에 넣어, 설치 걱정 없이 파이프라인에만 집중합니다. 코랩 런타임이 끊기면 처음부터 다시 실행하면 됩니다.

### 예제 구성
| 번호 | 예제 |
|---|---|
| 1 | 빈 상태에서 시작 — 대조군 |
| 2 | 최종 DAG — 매일 실행 · 재시도 · 적재 시각 |
| 3 | 스케줄러에 맡기기 — 스스로 한 번 도는 것 보기 |
| 4 | 문서를 넣으면 답이 바뀐다 |
| 5 | 장애 주입 — 임베딩 단계가 실패해도 |
| 6 | 내리는 순서 — 멈추고, 확인하고, 끈다 |

## 준비
칸이 다섯 개입니다. 위에서부터 차례로 한 번씩 실행합니다.

In [ ]:
# 준비 1 — Airflow 3.3.2 설치 (1~2분, 처음 한 번만)
import os, subprocess, sys, time

os.environ["TZ"] = "Asia/Seoul"          # 코랩 시계는 UTC — 화면의 시각을 한국 기준으로
time.tzset()

AF_VER = "3.3.2"                          # 이 책이 확인한 판
PY = f"{sys.version_info.major}.{sys.version_info.minor}"
HOME = "/content"
VENV = f"{HOME}/af"                       # 코랩 꾸러미와 섞이지 않게 가상환경에 설치합니다
PORT = 18080                              # 기본값은 8080 — 코랩이 8080 을 이미 써서 다른 번호를 씁니다

os.environ.update({
    "AIRFLOW_HOME": f"{HOME}/airflow",
    "AIRFLOW__CORE__DAGS_FOLDER": f"{HOME}/dags",      # DAG 파일을 두는 폴더
    "AIRFLOW__CORE__LOAD_EXAMPLES": "False",
    "AIRFLOW__CORE__PARALLELISM": "4",
    "AIRFLOW__API__PORT": str(PORT),
    "AIRFLOW__API__BASE_URL": f"http://localhost:{PORT}",
    "AIRFLOW__CORE__EXECUTION_API_SERVER_URL": f"http://localhost:{PORT}/execution/",
    "NO_COLOR": "1",                                    # 로그에 색 코드가 섞이지 않게
    "PATH": f"{VENV}/bin:" + os.environ["PATH"],
})
for d in ("dags", "data"):
    os.makedirs(f"{HOME}/{d}", exist_ok=True)

if not os.path.exists(f"{VENV}/bin/airflow"):
    print("설치 중입니다 …")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True, capture_output=True)
    subprocess.run([sys.executable, "-m", "uv", "venv", "-q", VENV, "--python", sys.executable], check=True)
    r = subprocess.run(
        [sys.executable, "-m", "uv", "pip", "install", "-q", "--python", f"{VENV}/bin/python",
         f"apache-airflow=={AF_VER}", "graphviz", 'fastembed', 'qdrant-client',
         "--constraint", f"https://raw.githubusercontent.com/apache/airflow/constraints-{AF_VER}/constraints-{PY}.txt"],
        capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:])
        raise SystemExit("설치하지 못했습니다. 이 칸을 한 번 더 실행해 주세요.")

r = subprocess.run("airflow db migrate", shell=True, capture_output=True, text=True)   # 메타데이터 DB(SQLite) 만들기
v = subprocess.run("airflow version", shell=True, capture_output=True, text=True).stdout.strip().splitlines()[-1]
print("설치 확인 · Airflow", v, "· 파이썬", PY)
print("DAG 폴더 ·", os.environ["AIRFLOW__CORE__DAGS_FOLDER"])

In [ ]:
# 준비 2 — 이 노트북에서 쓰는 도구 셋 (처음 한 번만)
import json, re
from datetime import datetime, timezone
from IPython.display import Image, display

_LOG = re.compile(r"^\d{4}-\d\d-\d\dT\S+ \[")
_ANSI = re.compile(r"\x1b\[[0-9;]*m")


def _sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return _ANSI.sub("", r.stdout + r.stderr)
_NOISE = ("graphviz", "You might need to install", "Please be reminded", "DagRun failed")


def af(args, show=True):
    """Airflow 명령을 부릅니다 — af("dags list") = airflow dags list. 로그 줄은 빼고 결과만 보여 줍니다."""
    keep = [l for l in _sh("airflow " + args).splitlines()
            if l.strip() and not _LOG.match(l) and not any(n in l for n in _NOISE)]
    text = "\n".join(keep)
    if show:
        print(text)
        return None                         # 화면에 찍었으면 돌려주지 않습니다(칸 아래에 한 번 더 나오지 않게)
    return text


def graph(dag_id):
    """DAG 모양을 그림으로 — 웹 UI 의 Graph 화면과 같은 것입니다."""
    png = f"{HOME}/data/{dag_id}.png"
    if os.path.exists(png):
        os.remove(png)
    af(f"dags show {dag_id} --save {png}", show=False)
    if not os.path.exists(png):             # 아직 등록 전인 새 파일이면 먼저 읽게 한 뒤 다시
        af("dags reserialize", show=False)
        af(f"dags show {dag_id} --save {png}", show=False)
    if os.path.exists(png):
        display(Image(png))
    else:                                   # 그림 도구가 없으면 화살표만 글자로
        print("\n".join(l.strip() for l in af(f"dags show {dag_id}", show=False).splitlines() if "->" in l))


def _utc(s):
    return datetime.fromisoformat(s).replace(tzinfo=timezone.utc)      # DB 의 시각은 UTC


def _kst(s):
    return _utc(s).astimezone().strftime("%H:%M:%S.%f")[:-3] if s else "-"


def run_dag(dag_id, conf=None):
    """DAG 를 한 번 끝까지 돌리고(airflow dags test), 태스크가 찍은 글과 태스크별 상태를 보여 줍니다."""
    cmd = f"airflow dags test {dag_id}" + (f" -c '{json.dumps(conf)}'" if conf else "")
    print(f"$ {cmd}\n실행 중입니다 …\n")
    out = _sh(cmd)
    task, tb = "-", []

    def flush():
        if tb:
            print(f"  [{task}] 오류 → {tb[-1].strip()}")
            tb.clear()

    for l in out.splitlines():
        m = re.search(r"\[DAG TEST\] starting task_id=(\S+)", l)
        if m:
            flush()
            task = m.group(1)
            continue
        if _LOG.match(l):
            flush()
            m = re.search(r"Done\. Returned value was: (.*?)\s+\[airflow", l)
            if m and m.group(1) != "None":
                print(f"  [{task}] 돌려준 값(XCom) → {m.group(1)}")
            m = re.search(r"Branch into (\S+)", l)
            if m:
                print(f"  [{task}] 고른 갈래 → {m.group(1)}")
            if "Skipping downstream tasks" in l and "ShortCircuitOperator" in l:
                print(f"  [{task}] False 를 돌려줘 뒤 태스크를 건너뜁니다")
            continue
        if not l.strip() or any(n in l for n in _NOISE):
            continue
        if tb or l.startswith("Traceback"):
            tb.append(l)
        else:
            print(f"  [{task}] {l}")
    flush()

    m = re.search(r"run_id=(manual__[^\s,]+)", out)
    if not m:                               # DAG 를 읽지 못한 경우 — 원인을 그대로 보여 줍니다
        print("\n".join(l for l in out.splitlines() if not _LOG.match(l))[-1500:])
        print("\nDAG 를 돌리지 못했습니다. 위 오류와 af(\"dags list-import-errors\") 를 확인해 주세요.")
        return {}
    import sqlite3                           # 태스크 상태는 메타데이터 DB 에서 바로 읽습니다(웹 UI 의 Grid 가 보는 곳)
    db = sqlite3.connect(f"{HOME}/airflow/airflow.db")
    rows = [dict(zip(("task_id", "state", "start_date", "end_date", "try_number"), r)) for r in db.execute(
        "select task_id, state, start_date, end_date, try_number from task_instance where dag_id=? and run_id=?",
        (dag_id, m.group(1)))]
    db.close()
    rows.sort(key=lambda x: x.get("start_date") or "9")
    print(f"\n{'태스크':<16}{'상태':<18}{'시도':<5}{'시작(한국 시각)':<16}걸린 시간")
    for x in rows:
        sec = ""
        if x.get("start_date") and x.get("end_date"):
            sec = f"{(_utc(x['end_date']) - _utc(x['start_date'])).total_seconds():.2f}초"
        print(f"{x['task_id']:<19}{(x['state'] or '-'):<20}{x['try_number']:<7}{_kst(x.get('start_date')):<20}{sec}")
    state = re.search(r"DagRun Finished:.*?state=(\w+)", out)
    print(f"\n실행 전체 상태 → {state.group(1) if state else '?'}")
    return {x["task_id"]: x["state"] for x in rows}


print("도구 준비 완료 · af( ) · graph( ) · run_dag( )")

In [ ]:
def start_airflow():
    """스케줄러까지 켭니다(airflow standalone). 20~40초 걸립니다."""
    import urllib.request
    global _SA
    if "_SA" in globals() and _SA.poll() is None:
        print("이미 켜져 있습니다")
        return
    _SA = subprocess.Popen("exec airflow standalone > /content/data/standalone.log 2>&1", shell=True,
                           start_new_session=True)      # 딸린 프로세스까지 한 묶음으로
    for _ in range(90):
        try:
            h = json.loads(urllib.request.urlopen(f"http://localhost:{PORT}/api/v2/monitor/health", timeout=3).read())
            if h["scheduler"]["status"] == "healthy" and h["dag_processor"]["status"] == "healthy":
                print("켜졌습니다 ·", " · ".join(f"{k} {v['status']}" for k, v in h.items()))
                return
        except Exception:
            pass
        time.sleep(2)
    print("3분 안에 켜지지 않았습니다. 로그의 끝부분입니다 ↓")
    print("\n".join(_ANSI.sub("", open("/content/data/standalone.log").read()).splitlines()[-15:]))


def stop_airflow():
    """스케줄러를 끕니다."""
    global _SA
    if "_SA" in globals() and _SA.poll() is None:
        import signal
        os.killpg(_SA.pid, signal.SIGTERM)
        try:
            _SA.wait(timeout=30)
        except Exception:
            os.killpg(_SA.pid, signal.SIGKILL)
    print("껐습니다")


print("도구 준비 완료 · start_airflow( ) · stop_airflow( )")

In [ ]:
def _db(sql, args=()):
    import sqlite3
    db = sqlite3.connect(f"{HOME}/airflow/airflow.db", timeout=30)
    rows = db.execute(sql, args).fetchall()
    db.close()
    return rows


def run_count(dag_id):
    return _db("select count(*) from dag_run where dag_id=?", (dag_id,))[0][0]


def wait_run(dag_id, more_than=0, seconds=240):
    """실행 수가 more_than 보다 많아지고 가장 최근 실행이 끝날 때까지 5초마다 상태를 찍습니다."""
    for i in range(seconds // 5):
        rows = _db("select run_id, state from dag_run where dag_id=? order by id desc", (dag_id,))
        if len(rows) > more_than:
            print(f"{i * 5:>3}초  {rows[0][0][:34]:36s} {rows[0][1]}")
            if rows[0][1] in ("success", "failed"):
                return rows[0][1]
        else:
            print(f"{i * 5:>3}초  (스케줄러가 실행을 만드는 중)")
        time.sleep(5)


def last_run(dag_id):
    """가장 최근 실행의 태스크별 상태 — 웹 UI 의 Grid 한 줄에 해당합니다."""
    run_id, state = _db("select run_id, state from dag_run where dag_id=? order by id desc limit 1", (dag_id,))[0]
    print(f"{run_id}  →  {state}\n")
    print(f"{'태스크':<16}{'상태':<18}{'시도':<5}{'시작(한국 시각)':<16}걸린 시간")
    rows = _db("select task_id, state, try_number, start_date, end_date from task_instance where dag_id=? and run_id=?",
               (dag_id, run_id))
    for task, st, tries, a, b in sorted(rows, key=lambda r: r[3] or "9"):
        sec = f"{(_utc(b) - _utc(a)).total_seconds():.2f}초" if a and b else ""
        print(f"{task:<19}{(st or '-'):<20}{tries:<7}{_kst(a):<20}{sec}")


print("도구 준비 완료 · wait_run( ) · last_run( ) · run_count( )")

In [ ]:
import shlex

for d in ("inbox", "lake/raw-docs"):
    os.makedirs(f"{HOME}/{d}", exist_ok=True)


def kb(cmd, *args):
    """kb.py 를 부릅니다 — kb("stat") · kb("search", "질문") · kb("ask", "질문") · kb("reset")"""
    line = f"{VENV}/bin/python {HOME}/dags/kb.py {cmd} " + " ".join(shlex.quote(str(a)) for a in args)
    out = _sh(line)
    keep = [l for l in out.splitlines() if l.strip() and "Fetching" not in l and "Warning" not in l and "warn" not in l]
    print("\n".join(keep))


def put(name, text):
    """투입구(inbox)에 문서를 넣습니다"""
    open(f"{HOME}/inbox/{name}", "w").write(text)
    print(f"투입구에 넣음 · {name} ({len(text)}자)")


print("도구 준비 완료 · kb( ) · put( )")

## 예제 1 · 빈 상태에서 시작 — 대조군

문서 세 개를 투입구에 넣고 지식베이스 도구를 만든 뒤, **아무것도 돌리지 않은 상태**에서 질문합니다.

In [ ]:
# 실습 문서 세 개 — 과정에서 다룬 도구의 짧은 안내문입니다
DOCS = {
    "airflow_ops.txt": """Airflow 운영 안내

Airflow 는 여러 단계로 된 작업을 정해진 순서와 시각에 실행하는 오케스트레이션 도구입니다. 작업의 순서도는 DAG 라고 부르고 파이썬 파일로 적습니다.

스케줄러가 멈추면 웹 화면은 열리지만 실행이 queued 상태에 머뭅니다. 이때는 docker compose ps 로 스케줄러 컨테이너의 상태를 보고 docker compose logs airflow-scheduler 로 원인을 확인합니다.

태스크가 실패하면 retries 에 적은 횟수만큼 다시 시도합니다. 그래도 실패하면 뒤 태스크는 upstream_failed 가 되어 시작하지 않습니다. 원인을 고친 뒤 실패한 태스크만 Clear 하면 그 자리부터 이어서 실행됩니다.

새 DAG 파일은 dags 폴더에 넣으면 dag-processor 가 읽어 등록합니다. 목록에 보이지 않으면 airflow dags list-import-errors 로 문법 오류가 있는지 먼저 확인합니다.
""",
    "minio_guide.txt": """MinIO 보관 안내

MinIO 는 S3 와 호환되는 오브젝트 저장소입니다. 우리 과정에서는 데이터 레이크의 저장소로 쓰며, 원본 문서를 고치지 않고 그대로 보관합니다.

문서는 버킷이라는 큰 통에 담습니다. 원본 문서는 raw-docs 버킷에 올리고, 같은 이름으로 다시 올리면 이전 파일을 덮어씁니다. 그래서 같은 문서를 여러 번 올려도 파일은 하나만 남습니다.

버킷의 내용은 mc ls 명령으로 확인합니다. 웹 콘솔은 9001 포트에서 열리고 프로그램은 9000 포트의 API 로 접속합니다.

원본을 Lake 에 먼저 보관해 두면 임베딩 방법이나 조각 크기를 바꾸고 싶을 때 원본에서 다시 만들 수 있습니다.
""",
    "qdrant_guide.txt": """Qdrant 검색 안내

Qdrant 는 벡터 데이터베이스입니다. 문서를 작은 조각으로 자르고 조각마다 임베딩 벡터를 만들어 넣어 두면, 질문과 뜻이 가까운 조각을 찾아 줍니다.

조각을 넣는 단위는 포인트이고 포인트의 묶음을 컬렉션이라고 부릅니다. 컬렉션을 만들 때 정한 벡터 차원과 넣는 벡터의 차원이 다르면 오류가 납니다.

포인트의 id 를 문서 이름과 조각 순번으로 만들면 같은 문서를 다시 넣어도 같은 자리에 덮어씁니다. 무작위 id 를 쓰면 넣을 때마다 조각이 중복으로 쌓입니다.

검색 결과에는 0 에서 1 사이의 유사도 점수가 붙습니다. 점수가 낮은 결과는 질문과 관계없는 조각이므로 기준 점수보다 낮으면 답하지 않는 것이 안전합니다.
""",
}

for name, text in DOCS.items():
    put(name, text)

In [ ]:
%%writefile /content/dags/kb.py
# kb.py — 지식베이스 도구
import sys
import uuid
from pathlib import Path

LAKE = Path("/content/lake/raw-docs")       # Lake — 원본 문서를 그대로 두는 곳
QDRANT = "/content/qdrant"                  # VectorDB — Qdrant(파일로 저장)
COLLECTION = "lecture_kb"
MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"   # 384차원 · 키 없이 코랩 안에서 계산
DIM = 384
THRESHOLD = 0.40                            # 이 점수보다 낮으면 「찾지 못했습니다」 — kb_02 에서 재서 정한 값

_model = None


def embed(texts):
    global _model
    if _model is None:
        from fastembed import TextEmbedding
        _model = TextEmbedding(MODEL, cache_dir="/content/fastembed")
    return [v.tolist() for v in _model.embed(list(texts))]


def chunk(text, size=200, overlap=40):
    """문단을 이어 붙여 size 글자쯤 되면 한 조각 — 조각 끝 overlap 글자는 다음 조각 앞에 다시 넣습니다."""
    chunks, buf = [], ""
    for para in [p.strip() for p in text.split("\n\n") if p.strip()]:
        if buf and len(buf) + len(para) > size:
            chunks.append(buf)
            buf = buf[-overlap:]
        buf = (buf + " " + para).strip()
    if buf:
        chunks.append(buf)
    return chunks


def client():
    from qdrant_client import QdrantClient, models
    c = QdrantClient(path=QDRANT)
    if not c.collection_exists(COLLECTION):
        c.create_collection(COLLECTION, vectors_config=models.VectorParams(size=DIM, distance=models.Distance.COSINE))
    return c


def _of(source):
    from qdrant_client import models
    return models.Filter(must=[models.FieldCondition(key="source", match=models.MatchValue(value=source))])


def remove_doc(c, source):
    from qdrant_client import models
    c.delete(COLLECTION, points_selector=models.FilterSelector(filter=_of(source)))


def index_doc(c, source, text, extra=None):
    """문서 하나를 넣습니다 — 먼저 그 문서의 조각을 지우고, 「문서 이름#순번」으로 만든 id 로 다시 넣습니다(멱등)."""
    from qdrant_client import models
    remove_doc(c, source)
    chunks = chunk(text)
    points = [models.PointStruct(id=str(uuid.uuid5(uuid.NAMESPACE_URL, f"{source}#{i}")), vector=v,
                                 payload={"source": source, "no": i, "text": t, **(extra or {})})
              for i, (t, v) in enumerate(zip(chunks, embed(chunks)))]
    c.upsert(COLLECTION, points=points)
    return len(points)


def stat(c):
    """문서별 조각 수"""
    out, offset = {}, None
    while True:
        points, offset = c.scroll(COLLECTION, limit=256, offset=offset, with_payload=True)
        for p in points:
            out[p.payload["source"]] = out.get(p.payload["source"], 0) + 1
        if offset is None:
            return dict(sorted(out.items()))


def search(question, k=3):
    c = client()
    try:
        hits = c.query_points(COLLECTION, query=embed([question])[0], limit=k, with_payload=True).points
        return [(round(h.score, 3), h.payload["source"], h.payload["text"]) for h in hits]
    finally:
        c.close()


def answer(question, threshold=THRESHOLD):
    """가장 가까운 조각의 점수가 기준보다 낮으면 답하지 않습니다."""
    hits = search(question)
    if not hits or hits[0][0] < threshold:
        top = hits[0][0] if hits else 0
        return f"찾지 못했습니다 (가장 가까운 조각의 점수 {top} < 기준 {threshold})"
    score, source, text = hits[0]
    return f"[근거 {source} · 점수 {score}]\n{text}"


if __name__ == "__main__":                  # 터미널에서: python kb.py stat | search "질문" | ask "질문" | reset
    cmd = sys.argv[1] if len(sys.argv) > 1 else "stat"
    if cmd == "reset":
        import shutil
        shutil.rmtree(QDRANT, ignore_errors=True)
        print("지식베이스를 비웠습니다")
    elif cmd == "stat":
        c = client()
        s = stat(c)
        c.close()
        print(f"조각 {sum(s.values())}개 · 문서 {len(s)}개")
        for name, n in s.items():
            print(f"  {name:20s} {n}조각")
    elif cmd == "search":
        for score, source, text in search(sys.argv[2]):
            print(f"  {score:<6} {source:18s} {text[:46]}…")
    elif cmd == "ask":
        print(answer(sys.argv[2], float(sys.argv[3]) if len(sys.argv) > 3 else THRESHOLD))

In [ ]:
kb("reset")
kb("ask", "스케줄러가 멈추면 어떻게 되나요?")

***확인 포인트***: 「찾지 못했습니다 (가장 가까운 조각의 점수 0 < 기준 0.4)」가 나옵니다. kb_02 에서 만든 답하기는 그대로인데 지식베이스가 비어 있습니다. (첫 실행은 임베딩 모델을 내려받느라 30초쯤 걸립니다.)

## 예제 2 · 최종 DAG — 매일 실행 · 재시도 · 적재 시각

kb_01 의 다섯 태스크(collect → validate → build_index → prune → report)에 운영에 필요한 네 가지를 더합니다.

| 더한 것 | 왜 |
|---|---|
| `schedule="@daily"` | 사람이 돌리지 않아도 매일 갱신 |
| `retries=2` · `retry_delay` 10초 | 임베딩 서버가 잠깐 응답하지 않아도 스스로 회복 |
| `max_active_runs=1` | 두 실행이 지식베이스를 동시에 고치지 않게 |
| 조각마다 `ingested_at` | 이 답의 근거가 **언제 들어온 것인지** 추적 |

스케줄러에 맡기기 전에 `run_dag` 로 한 번 끝까지 돌려 DAG 가 맞는지부터 봅니다. 개발할 때는 늘 이 순서입니다.

In [ ]:
%%writefile /content/dags/rag_pipeline_final.py
# rag_pipeline_final.py — kb_01 의 파이프라인 + 매일 실행 · 재시도 · 적재 시각
import shutil
from datetime import datetime, timedelta
from pathlib import Path

from airflow.sdk import DAG
from airflow.providers.standard.operators.python import PythonOperator

import kb

INBOX = Path("/content/inbox")
FAIL_ONCE = Path("/content/data/fail_once")     # 이 파일이 있으면 임베딩 단계가 한 번 실패합니다(장애 연습용)
MIN_CHARS = 50


def collect():
    kb.LAKE.mkdir(parents=True, exist_ok=True)
    names = sorted(p.name for p in INBOX.glob("*.txt"))
    for name in names:
        shutil.copyfile(INBOX / name, kb.LAKE / name)
    gone = sorted(q.name for q in kb.LAKE.glob("*.txt") if q.name not in names)
    for name in gone:
        (kb.LAKE / name).unlink()
    print(f"Lake 에 올린 문서 {len(names)}개 · 내린 문서 {gone}")
    return names


def validate():
    bad = [f"{p.name}({len(p.read_text().strip())}자)" for p in sorted(kb.LAKE.glob("*.txt"))
           if len(p.read_text().strip()) < MIN_CHARS]
    if bad:
        raise ValueError(f"너무 짧은 문서가 있습니다 → {bad}")
    print("모든 문서 통과")


def build_index(ti):
    if FAIL_ONCE.exists():
        FAIL_ONCE.unlink()
        raise ConnectionError(f"임베딩 서버 응답 없음({ti.try_number}번째 시도 · 일부러 낸 오류)")
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    c = kb.client()
    try:
        total = 0
        for p in sorted(kb.LAKE.glob("*.txt")):
            total += kb.index_doc(c, p.name, p.read_text(), extra={"ingested_at": stamp})   # 언제 넣었는지 남깁니다
        print(f"{ti.try_number}번째 시도에 성공 · {total}조각 · 적재 시각 {stamp}")
        return total
    finally:
        c.close()


def prune():
    keep = {q.name for q in kb.LAKE.glob("*.txt")}
    c = kb.client()
    try:
        gone = [name for name in kb.stat(c) if name not in keep]
        for name in gone:
            kb.remove_doc(c, name)
        print(f"지운 문서 → {gone}")
    finally:
        c.close()


def report():
    c = kb.client()
    try:
        s = kb.stat(c)
    finally:
        c.close()
    print(f"지식베이스 전체 {sum(s.values())}조각 · 문서 {len(s)}개")


with DAG(
    dag_id="rag_pipeline_final",
    start_date=datetime(2026, 1, 1),
    schedule="@daily",                      # 매일 한 번
    catchup=False,                          # 밀린 날짜는 돌리지 않습니다
    max_active_runs=1,                      # 두 실행이 지식베이스를 동시에 고치지 않게
    default_args={"retries": 2, "retry_delay": timedelta(seconds=10)},
    tags=["week8", "rag", "final"],
) as dag:
    t = {f.__name__: PythonOperator(task_id=f.__name__, python_callable=f)
         for f in (collect, validate, build_index, prune, report)}

    t["collect"] >> t["validate"] >> t["build_index"] >> t["prune"] >> t["report"]

In [ ]:
s = run_dag("rag_pipeline_final")
graph("rag_pipeline_final")

In [ ]:
kb("stat")
print()
kb("ask", "스케줄러가 멈추면 어떻게 되나요?")

***확인 포인트***: 다섯 태스크가 모두 success 이고 조각 10개 · 문서 3개가 들어갑니다. 같은 질문에 이번에는 `[근거 airflow_ops.txt · 점수 0.476]` 과 함께 답이 나옵니다.

**바꿔 보기**
- `schedule="@daily"` 는 한국 시각으로 몇 시일까요? Airflow 의 기본 시간대는 UTC 입니다. `af("dags next-execution rag_pipeline_final")` 로 확인합니다(출력은 UTC — 한국 시각은 9시간을 더합니다).

## 예제 3 · 스케줄러에 맡기기 — 스스로 한 번 도는 것 보기

지금까지는 우리가 직접 돌렸습니다. 이제 스케줄러를 켜고 DAG 의 일시정지를 풉니다. **실행을 요청하지 않습니다.**

In [ ]:
start_airflow()
af("dags reserialize", show=False)

n = run_count("rag_pipeline_final")
af("dags unpause rag_pipeline_final", show=False)
print("일시정지를 풀었습니다 — 실행 요청은 하지 않았습니다\n")
state = wait_run("rag_pipeline_final", more_than=n)

In [ ]:
last_run("rag_pipeline_final")

***확인 포인트***: 아무도 trigger 하지 않았는데 실행이 하나 생겨 `success` 로 끝납니다. 실행 이름이 `manual__…` 이 아니라 **`scheduled__…`** 입니다 — 스케줄러가 만든 실행입니다.

`catchup=False` 인 DAG 를 켜면 스케줄러는 「가장 최근에 돌았어야 할 한 번」만 바로 돌립니다. 내일부터는 매일 그 시각에 스스로 돕니다.

**바꿔 보기**
- 위 `last_run` 칸을 다시 실행해도 결과가 같습니다. 이 표는 메타데이터 DB 에서 읽은 것이라, 노트북을 닫았다 열어도 이력이 남아 있습니다.

## 예제 4 · 문서를 넣으면 답이 바뀐다

이 과정 전체가 향해 온 장면입니다. 지식베이스에 없는 것을 묻고 → 문서를 투입구에 넣고 → 파이프라인이 돌고 → 같은 질문을 다시 합니다.
매일 자정까지 기다릴 수 없으니 이번 한 번만 실행을 요청합니다(`dags trigger`).

In [ ]:
print("── 문서를 넣기 전")
kb("ask", "레이크하우스가 뭔가요?")

In [ ]:
LAKEHOUSE = """레이크하우스 안내

레이크하우스는 데이터 레이크와 데이터 웨어하우스의 장점을 합친 구조입니다. 값싼 오브젝트 저장소에 파일을 두면서도 표처럼 다룰 수 있습니다.

우리 과정에서는 MinIO 위에 Iceberg 표 형식을 얹어 레이크하우스를 만들었습니다. Iceberg 는 어떤 파일이 표에 속하는지와 표의 변경 이력을 기록합니다.

같은 Iceberg 표를 Spark 와 Trino 두 엔진으로 읽을 수 있습니다. Spark 로 적재하고 Trino 로 빠르게 조회하는 식으로 역할을 나눕니다.

스냅샷이 남기 때문에 과거 시점의 데이터를 다시 조회할 수 있고, 잘못 적재했을 때 이전 스냅샷으로 되돌릴 수 있습니다.
"""

put("lakehouse.txt", LAKEHOUSE)

n = run_count("rag_pipeline_final")
af("dags trigger rag_pipeline_final", show=False)
print("실행을 요청했습니다\n")
state = wait_run("rag_pipeline_final", more_than=n)

In [ ]:
print("── 문서를 넣은 뒤")
kb("ask", "레이크하우스가 뭔가요?")
print()
kb("stat")

***확인 포인트***: 넣기 전에는 「찾지 못했습니다 (… 0.372 < 기준 0.4)」, 넣은 뒤에는 **`[근거 lakehouse.txt · 점수 0.622]`** 와 함께 답이 나옵니다. 조각은 10 → 12개입니다.

챗봇 코드는 한 글자도 고치지 않았습니다. 바뀐 것은 투입구의 파일 하나이고, 그 파일이 Lake → 검증 → 임베딩 → VectorDB 를 거쳐 답이 된 것입니다.
이 책에서 따로 만든 Loader · 청킹 · 검색 · 답하기가 여기서 한 줄로 이어집니다.

**바꿔 보기**
- 내 업무 문서로 해 봅니다. `put("my_doc.txt", """…""")` → 위 칸의 trigger · wait_run 을 다시 실행 → 그 문서에만 있는 내용을 `kb("ask", "…")` 로 묻습니다.

## 예제 5 · 장애 주입 — 임베딩 단계가 실패해도

운영 중에는 임베딩 서버가 잠깐 응답하지 않는 일이 생깁니다. `fail_once` 파일을 만들어 `build_index` 가 **한 번** 실패하게 한 뒤 실행을 요청합니다. 아무도 손대지 않습니다.

In [ ]:
open("/content/data/fail_once", "w").close()
print("장애를 심었습니다 — build_index 가 한 번 실패합니다\n")

n = run_count("rag_pipeline_final")
af("dags trigger rag_pipeline_final", show=False)
state = wait_run("rag_pipeline_final", more_than=n)
print()
last_run("rag_pipeline_final")
print()
kb("stat")

***확인 포인트***: 실행은 **success** 로 끝나고, 표에서 `build_index` 의 **시도가 2** 입니다. 첫 시도가 실패하자 10초 쉬고 다시 해서 성공한 것입니다. 다른 태스크는 시도 1 그대로이고, 지식베이스는 12조각 그대로입니다.

재시도가 안전했던 이유는 kb_01 에서 확인한 멱등성입니다. `build_index` 를 두 번 실행해도 조각이 늘지 않으니, 실패한 자리에서 그냥 다시 하면 됩니다. **멱등하지 않은 태스크에 retries 를 주면 재시도가 사고가 됩니다.**

**바꿔 보기**
- 장애가 재시도 횟수보다 길면 어떻게 될까요? `build_index` 에서 `FAIL_ONCE.unlink()` 줄을 주석 처리해 「계속 실패」로 만들고(파일을 다시 만든 뒤) 돌려 봅니다. 시도 3 까지 간 뒤 failed, 뒤 태스크는 upstream_failed 가 됩니다. 그동안 지식베이스는 어떤 상태인가요? 확인한 뒤 코드를 되돌리고 `fail_once` 파일을 지웁니다.

## 예제 6 · 내리는 순서 — 멈추고, 확인하고, 끈다

내리는 데에도 순서가 있습니다. DAG 파일을 먼저 지우면 워커가 읽을 파일이 없어 실행 중인 것이 깨집니다. 내릴 때는 **새 실행을 막고 → 도는 것이 없는지 보고 → 끕니다.**

In [ ]:
af("dags pause rag_pipeline_final", show=False)              # ① 새 실행이 생기지 않게
print("① 일시정지\n")

print("② 실행 이력 — running 이 없는지 확인")
for run_id, state, kind in _db("select run_id, state, run_type from dag_run where dag_id=? order by id",
                               ("rag_pipeline_final",)):
    print(f"   {run_id[:36]:38s} {kind:10s} {state}")

print()
stop_airflow()                                               # ③ 서비스 끄기

In [ ]:
# 서비스가 꺼져도 지식베이스는 남아 있습니다 — 챗봇은 계속 답합니다
kb("ask", "레이크하우스가 뭔가요?")

***확인 포인트***: 실행 이력에 네 줄이 있습니다 — 예제 2 의 `manual`(run_dag), 예제 3 의 **`scheduled`**, 예제 4 · 5 의 `manual` 둘. 모두 success 이고 running 은 없습니다.
서비스를 끈 뒤에도 질문에 답이 나옵니다. Airflow 는 지식베이스를 **갱신하는** 쪽이고, 답하는 쪽은 지식베이스만 있으면 됩니다. 파이프라인이 멈춰도 챗봇은 「어제까지의 지식」으로 계속 일합니다.

**바꿔 보기**
- 반대의 경우를 생각해 봅니다. 파이프라인은 도는데 지식베이스(VectorDB)가 내려가면 어느 태스크가 실패하고, 재시도는 도움이 될까요?

## 정리 — 한 줄로 이어진 것

| 책의 어디 | 만든 것 | 이 노트북에서 맡은 자리 |
|---|---|---|
| 5장 Loader | 받은 자료를 문서로, 출처와 시각 | `collect` 가 올리는 문서 |
| 6장 청킹 | 조각 크기를 표로 고르기 | `index_doc` 이 문서를 자르는 크기 |
| 4장 · 17장 검색과 기준 점수 | 찾고, 모르면 모른다고 답하기 | `kb("ask", …)` |
| 16장 파이프라인 | 보관 → 검증 → 임베딩 → 빼기 | 다섯 태스크 |
| 18장 스케줄과 재시도 | 매일, 실패해도 스스로 | `@daily` · `retries` |

## 마무리
이 노트북은 「코랩으로 만드는 업무 챗봇」 18장의 예제입니다. 「바꿔 보기」에서 고친 것과 달라진 결과를 한두 줄로 적어 두면, 내 문서로 바꿀 때 길잡이가 됩니다.